# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IbtissamToure/FlyRank-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Ranked actions + reason codes

The action queue ranks content pages by the Logistic Regression model score, helping the team identify which pages may deserve human review first.

Each recommendation includes transparent reason codes:
- `high_model_score`: the page is among the highest-scoring pages.
- `high_search_visibility`: the page has at least 3,000 impressions, making it relevant for review prioritization.
- `stale_content`: the time since the last update is in the top quartile of the dataset.

The score is a prioritization signal, not a confirmed probability of decline or a guarantee that updating the page will improve its performance.

In [2]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Load and prepare the dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df = df[df["trend_direction"] != "new"].copy()

y = (df["trend_direction"] == "down").astype(int)

numeric_features = [
    "search_volume", "competition", "cpc",
    "word_count", "char_count", "content_age_days",
    "days_since_last_update"
]

categorical_features = [
    "competition_level", "content_type", "main_intent",
    "provider_used", "model_used", "age_tier",
    "freshness_tier", "word_count_tier", "char_count_tier"
]

feature_columns = numeric_features + categorical_features
X = df[feature_columns]

# Rebuild the Week 5 model pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Fit the model and generate ranking scores
model.fit(X, y)

queue = df[
    ["content_id", "impressions_90d", "days_since_last_update"]
].copy()

queue["model_score"] = model.predict_proba(X)[:, 1]

# Define reason-code thresholds
high_score_threshold = queue["model_score"].quantile(0.90)
stale_threshold = queue["days_since_last_update"].quantile(0.75)

def assign_reason_codes(row):
    reasons = []

    if row["model_score"] >= high_score_threshold:
        reasons.append("high_model_score")

    if row["impressions_90d"] >= 3000:
        reasons.append("high_search_visibility")

    if row["days_since_last_update"] >= stale_threshold:
        reasons.append("stale_content")

    return ", ".join(reasons) if reasons else "standard_review"

queue["reason_codes"] = queue.apply(assign_reason_codes, axis=1)

queue = queue.sort_values(
    "model_score", ascending=False
).reset_index(drop=True)

queue.insert(0, "priority_rank", range(1, len(queue) + 1))

display(queue.head(20))

print("Total pages ranked:", len(queue))
print("High-score threshold:", round(high_score_threshold, 3))
print("Stale-content threshold:", round(stale_threshold, 1))

,priority_rank,content_id,impressions_90d,days_since_last_update,model_score,reason_codes
0,1,content_6b56a0830317,27,104,0.803704,"high_model_score, stale_content"
1,2,content_330a9f2d60d6,26,104,0.798841,"high_model_score, stale_content"
2,3,content_ed3cebd8dc66,14,8,0.797619,high_model_score
3,4,content_760805dfb402,51,8,0.794690,high_model_score
4,5,content_de7526789916,19,104,0.792466,"high_model_score, stale_content"
5,6,content_cd89c08a4fce,63,8,0.792420,high_model_score
6,7,content_3bd08a2d2168,1,8,0.791874,high_model_score
7,8,content_c865ed29b025,4,8,0.788321,high_model_score
8,9,content_dec8916cf762,3,8,0.783056,high_model_score
9,10,content_ca579e81217b,5,8,0.782717,high_model_score


Total pages ranked: 27764
High-score threshold: 0.63
Stale-content threshold: 104.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended use and limits

### Intended users
SEO analysts and content strategists responsible for reviewing and maintaining content portfolios.

### Intended use
The playbook provides a decision-support queue that ranks pages for human review using the Logistic Regression model score and transparent reason codes.

Analysts can use the queue to:
- Identify pages with high model scores.
- Review stale content.
- Consider search visibility before prioritizing an update.

### Limitations
- The model identifies patterns associated with historical declining labels; it does not establish causation.
- The current queue is generated by fitting the model on the available labeled dataset, so its ranking is not an independent performance evaluation.
- The model was evaluated using a grouped client split, but temporal generalization has not been established.
- Model scores should not be interpreted as calibrated probabilities.
- The starter dataset represents a limited portfolio and may not generalize to other clients or periods.

The final decision to update, retain, or remove a page must remain with a human analyst.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Summarize the scope and limitations of the action queue

playbook_summary = pd.DataFrame([
    {
        "item": "Intended users",
        "description": "SEO analysts and content strategists"
    },
    {
        "item": "Purpose",
        "description": "Prioritize pages for human review"
    },
    {
        "item": "Model evidence",
        "description": "60% Precision@50 on the grouped test set"
    },
    {
        "item": "Baseline",
        "description": "44% Precision@50 on the same test population"
    },
    {
        "item": "Generalization",
        "description": "Future performance has not been established"
    },
    {
        "item": "Decision authority",
        "description": "Final actions remain with human analysts"
    }
])

display(playbook_summary)

,item,description
0,Intended users,SEO analysts and content strategists
1,Purpose,Prioritize pages for human review
2,Model evidence,60% Precision@50 on the grouped test set
3,Baseline,44% Precision@50 on the same test population
4,Generalization,Future performance has not been established
5,Decision authority,Final actions remain with human analysts


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human review + the no-go list

### Required human review

Before taking action on a recommended page, an analyst must:

1. Verify the page's current search performance and traffic trends.
2. Check whether the content is outdated, inaccurate, or incomplete.
3. Review search intent and whether the page still serves its intended audience.
4. Check for recent updates or external factors that may explain the model score.
5. Decide whether to update, monitor, or leave the page unchanged.

### No-go list

The model must never:

- Automatically publish, rewrite, delete, or redirect content.
- Make business-critical decisions without human approval.
- Treat a high score as proof that a page is declining.
- Recommend changes based only on model scores without reviewing the underlying page.
- Be used as a substitute for SEO expertise or current performance evidence.

Every recommendation is a review signal, not an instruction to make an immediate change.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Document mandatory human-review controls

human_review_controls = pd.DataFrame([
    {
        "control": "Performance verification",
        "requirement": "Check current search performance and traffic trends"
    },
    {
        "control": "Content quality",
        "requirement": "Review accuracy, completeness, and freshness"
    },
    {
        "control": "Search intent",
        "requirement": "Confirm the page still matches user intent"
    },
    {
        "control": "Context check",
        "requirement": "Investigate external factors and recent updates"
    },
    {
        "control": "Final approval",
        "requirement": "Human analyst decides the appropriate action"
    }
])

no_go_actions = [
    "Automatic publishing or rewriting",
    "Automatic deletion or redirection",
    "Unreviewed business-critical decisions",
    "Treating model scores as proof of decline"
]

display(human_review_controls)

print("\nActions prohibited from automation:")
for action in no_go_actions:
    print("-", action)

,control,requirement
0,Performance verification,Check current search performance and traffic t...
1,Content quality,"Review accuracy, completeness, and freshness"
2,Search intent,Confirm the page still matches user intent
3,Context check,Investigate external factors and recent updates
4,Final approval,Human analyst decides the appropriate action



Actions prohibited from automation:
- Automatic publishing or rewriting
- Automatic deletion or redirection
- Unreviewed business-critical decisions
- Treating model scores as proof of decline


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / retrain triggers

The action queue should be monitored regularly to ensure that its recommendations remain relevant and useful.

### Monitoring indicators

- **Precision@50:** Track whether the highest-ranked pages continue to contain a meaningful proportion of declining pages.
- **Base rate:** Monitor changes in the proportion of declining pages.
- **Feature drift:** Check whether input distributions change significantly over time.
- **Reason-code distribution:** Monitor changes in the number of pages assigned each reason code.
- **Human feedback:** Record whether analysts consider recommendations useful and appropriate.

### Retraining triggers

Consider reviewing or retraining the model when:

- Precision@50 declines consistently across evaluation periods.
- The decline base rate changes substantially.
- Feature distributions shift significantly.
- New content types or client populations appear.
- Analysts repeatedly reject recommendations as irrelevant.

Retraining should only follow a new evaluation using an appropriate grouped or time-based validation design. A change in the data alone does not automatically justify retraining.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define monitoring metrics and retraining triggers

monitoring_plan = pd.DataFrame([
    {
        "metric": "Precision@50",
        "purpose": "Track ranking quality",
        "trigger": "Consistent decline across evaluation periods"
    },
    {
        "metric": "Decline base rate",
        "purpose": "Monitor changes in label distribution",
        "trigger": "Substantial change in decline rate"
    },
    {
        "metric": "Feature drift",
        "purpose": "Detect changes in input distributions",
        "trigger": "Significant distribution shifts"
    },
    {
        "metric": "Reason-code distribution",
        "purpose": "Track changes in recommendation patterns",
        "trigger": "Unexpected changes in reason-code frequency"
    },
    {
        "metric": "Analyst feedback",
        "purpose": "Assess practical usefulness",
        "trigger": "Repeated rejection of recommendations"
    }
])

display(monitoring_plan)

print("\nCurrent reference metrics:")
print("Precision@50:", 0.600)
print("Baseline Precision@50:", 0.440)
print("Test base rate:", 0.434)

,metric,purpose,trigger
0,Precision@50,Track ranking quality,Consistent decline across evaluation periods
1,Decline base rate,Monitor changes in label distribution,Substantial change in decline rate
2,Feature drift,Detect changes in input distributions,Significant distribution shifts
3,Reason-code distribution,Track changes in recommendation patterns,Unexpected changes in reason-code frequency
4,Analyst feedback,Assess practical usefulness,Repeated rejection of recommendations



Current reference metrics:
Precision@50: 0.6
Baseline Precision@50: 0.44
Test base rate: 0.434


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## 5. Exports for the paper

The final action queue is exported as a CSV file for further review and use in the research paper.

The export includes:
- Priority rank
- Content identifier
- Search impressions
- Days since last update
- Model score
- Reason codes

The queue is a decision-support artifact generated from the available dataset. It is not an automated content-update plan or an independently validated future prediction.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path

# Create the output directory
output_dir = Path("../../work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Export the ranked action queue
queue_path = output_dir / "w07_ranked_action_queue.csv"
queue.to_csv(queue_path, index=False)

# Export the monitoring plan
monitoring_path = output_dir / "w07_monitoring_plan.csv"
monitoring_plan.to_csv(monitoring_path, index=False)

print("Exports completed:")
print(queue_path)
print(monitoring_path)

print("\nExport summary:")
print("Queue rows:", len(queue))
print("Queue columns:", len(queue.columns))
print("Monitoring metrics:", len(monitoring_plan))

Exports completed:
../../work/outputs/w07_ranked_action_queue.csv
../../work/outputs/w07_monitoring_plan.csv

Export summary:
Queue rows: 27764
Queue columns: 6
Monitoring metrics: 5


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.